In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l0.3 Probabilities

> 🎯 **Goal:** look inside the model at the single most important thing it produces, a probability distribution over the next character, and chart it.

**Why this matters.** It is tempting to think a language model "decides" what comes next. It does not. At every step it does something humbler and more interesting: it assigns a probability to every possible next character and hands you the whole list. The actual choosing happens afterward (that is l0.4). Understanding this split, the model proposes a distribution, decoding disposes, is the single biggest conceptual unlock in this unit. Once you see it, everything about temperature, sampling, and "why did it say that" clicks into place.

**The intuition.** Give the model the prefix `ROME` and instead of answering "O" it answers something like: "75 percent O, 8 percent S, 5 percent newline, ... " covering every character it knows, with the numbers adding up to 1 (that is, to 100 percent). It is a weather forecast for the next letter. A confident, well-trained model puts almost all the probability on one character; an unsure model spreads it out. The bar chart below is literally a picture of the model's belief.

**The mechanics.** A *probability distribution* over the vocabulary is a list of non-negative numbers, one per character, that sum to 1. The function `next_token_probs` runs the model on a prefix and returns exactly that, as a dictionary mapping each character to its probability. We sort it, take the top 10, and draw a bar for each. (We set Matplotlib to a headless "Agg" backend first, because the CI machine has no screen to draw on.) The height of each bar is how strongly the model expects that character next.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless backend (CI has no display)
import matplotlib.pyplot as plt
from pocketlm import train_tiny, pick_config, next_token_probs

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
probs = next_token_probs(model, tok, "ROME")
top = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)[:10]
plt.bar([repr(c) for c, _ in top], [p for _, p in top])
plt.title("Top-10 next-char probabilities after 'ROME'")
plt.show()

**What just happened.** You charted the model's mind. Each bar is its belief about the next character after `ROME`. A well-trained model would put most of the mass on `'O'` (spelling ROMEO); our barely-trained one may be fuzzier, but the shape of the idea is the same. Crucially, no single character has probability 1 and none is exactly 0: the model is hedging across many options, and that hedge is the raw material decoding works with next lesson.

⚠️ **Common pitfalls**
- Reading the tallest bar as "the answer". It is only the *most likely* answer. The model never commits here; it only ranks.
- Forgetting the chart is the top 10 of a much longer list. Every character in the vocabulary gets some probability, even the tiny ones not shown.

🏋️ **Try it yourself.** Change the prefix `"ROME"` to `"ROMEO:"` or `"To be, or not to b"` and re-run. Watch how a more familiar prefix produces a sharper, more confident chart (one tall bar) while an unusual prefix flattens it out.

The whole distribution, not just the top 10, always sums to 1, which the assert confirms:

In [ ]:
assert abs(sum(probs.values()) - 1.0) < 1e-5